### Ingest the 4 KPI CSVs into Bronze Delta tables

In [0]:
%sql
USE CATALOG policyiq;

CREATE OR REPLACE TABLE bronze.branch_dim AS
SELECT * FROM read_files(
  '/Volumes/policyiq/sources/kpi_raw_files/branch_dim.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE bronze.policy_registry AS
SELECT * FROM read_files(
  '/Volumes/policyiq/sources/kpi_raw_files/policy_registry_seed.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE bronze.kpi_registry AS
SELECT * FROM read_files(
  '/Volumes/policyiq/sources/kpi_raw_files/kpi_registry_seed.csv',
  format => 'csv', header => true, inferSchema => true
);

CREATE OR REPLACE TABLE bronze.kpi_actuals AS
SELECT * FROM read_files(
  '/Volumes/policyiq/sources/kpi_raw_files/kpi_actuals.csv',
  format => 'csv', header => true, inferSchema => true
);

In [0]:
%sql
SELECT 'branch_dim' t, count(*) FROM bronze.branch_dim
UNION ALL SELECT 'policy_registry', count(*) FROM bronze.policy_registry
UNION ALL SELECT 'kpi_registry', count(*) FROM bronze.kpi_registry
UNION ALL SELECT 'kpi_actuals', count(*) FROM bronze.kpi_actuals;

### Land the raw PDF bytes into a Bronze table

In [0]:
%sql
CREATE OR REPLACE TABLE bronze.policy_documents_raw AS
SELECT
  path,
  regexp_extract(path, '([^/]+)$', 1)      AS file_name,
  content,
  length,
  modificationTime
FROM read_files(
  '/Volumes/policyiq/sources/policy_pdf/',
  format => 'binaryFile'
);

SELECT file_name, length FROM bronze.policy_documents_raw ORDER BY file_name;

### Map each file to its policy_id

In [0]:
%sql
CREATE OR REPLACE TABLE bronze.policy_documents_mapped AS
SELECT
  *,
  CASE
    WHEN lower(file_name) LIKE '%ekyc%'                          THEN 'EKYC_2026'
    WHEN lower(file_name) LIKE '%aml%'                           THEN 'AML_MLTF_2026'
    WHEN lower(file_name) LIKE '%credit%'                        THEN 'CRM_2016'
    WHEN lower(file_name) LIKE '%cyber%'                         THEN 'CYBERSEC_2026'
    WHEN lower(file_name) LIKE '%hr%' OR lower(file_name) LIKE '%leave%' THEN 'HR_LEAVE_2026'
    ELSE 'UNKNOWN'
  END AS policy_id
FROM bronze.policy_documents_raw;

SELECT file_name, policy_id FROM bronze.policy_documents_mapped ORDER BY policy_id;

### Parse all 5 PDFs

In [0]:
%sql
CREATE OR REPLACE TABLE bronze.policy_documents_parsed AS
SELECT
  file_name,
  policy_id,
  ai_parse_document(content, map('version', '2.0')) AS parsed
FROM bronze.policy_documents_mapped;

In [0]:
%sql
SELECT
  file_name,
  policy_id,
  size(from_json(to_json(parsed:document.pages), 'array<variant>')) AS page_count
FROM bronze.policy_documents_parsed
ORDER BY policy_id;

In [0]:
%sql
SELECT 'branch_dim' AS dataset, count(*) AS row_count FROM bronze.branch_dim
UNION ALL SELECT 'policy_registry', count(*) FROM bronze.policy_registry
UNION ALL SELECT 'kpi_registry', count(*) FROM bronze.kpi_registry
UNION ALL SELECT 'kpi_actuals', count(*) FROM bronze.kpi_actuals
UNION ALL SELECT 'policy_documents_parsed', count(*) FROM bronze.policy_documents_parsed;
